Data Exploration

In [2]:
# 01_data_exploration.py
import pandas as pd
import numpy as np

# ── Load ──────────────────────────────────────────────────────────────
df = pd.read_csv("loan.csv", nrows=100_000, low_memory=False)



In [3]:
print(f"Shape: {df.shape}")                    # rows × columns
print(f"\nColumns:\n{df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes.value_counts()}")
print(f"\nMissing values:\n{df.isna().sum()}")

Shape: (100000, 145)

Columns:
['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annual_inc_joint', 'dti_joint', 'verification_status_joint', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'open_acc_6m', 'open_

In [4]:
# ── Look at the target variable ────────────────────────────────────────
# loan_status is our raw target — we need to convert it to binary
print("\nloan_status value counts:")
print(df['loan_status'].value_counts())


loan_status value counts:
loan_status
Current               96793
Fully Paid             2431
Late (31-120 days)      318
In Grace Period         316
Late (16-30 days)       123
Charged Off              19
Name: count, dtype: int64


In [5]:
# ── Check missingness ──────────────────────────────────────────────────
missing = df.isnull().mean().sort_values(ascending=False)
print("\nTop 20 columns by % missing:")
print(missing[missing > 0].head(20))


Top 20 columns by % missing:
id                                            1.00000
desc                                          1.00000
url                                           1.00000
member_id                                     1.00000
hardship_length                               0.99999
hardship_reason                               0.99999
hardship_status                               0.99999
deferral_term                                 0.99999
hardship_amount                               0.99999
hardship_start_date                           0.99999
hardship_end_date                             0.99999
payment_plan_start_date                       0.99999
hardship_dpd                                  0.99999
hardship_loan_status                          0.99999
orig_projected_additional_accrued_interest    0.99999
hardship_payoff_balance_amount                0.99999
hardship_last_payment_amount                  0.99999
debt_settlement_flag_date                     0.9999

In [6]:
# ── Quick stats on key numeric columns ────────────────────────────────
key_cols = ['loan_amnt', 'int_rate', 'dti', 'annual_inc',
            'installment', 'revol_util', 'open_acc', 'total_acc']
print("\nDescriptive stats:")
print(df[key_cols].describe())


Descriptive stats:
           loan_amnt       int_rate           dti    annual_inc  \
count  100000.000000  100000.000000  99818.000000  1.000000e+05   
mean    15934.297500      13.004960     19.891596  8.314996e+04   
std     10146.154617       5.026647     19.164748  1.117613e+05   
min      1000.000000       6.000000      0.000000  0.000000e+00   
25%      8000.000000       8.810000     11.770000  4.784000e+04   
50%     14000.000000      11.800000     17.980000  6.861300e+04   
75%     21500.000000      16.140000     25.300000  1.000000e+05   
max     40000.000000      30.990000    999.000000  9.757200e+06   

         installment    revol_util       open_acc      total_acc  
count  100000.000000  99879.000000  100000.000000  100000.000000  
mean      462.493739     44.328556      11.571810      22.686730  
std       285.647728     24.883297       5.984635      12.117233  
min        30.640000      0.000000       0.000000       2.000000  
25%       252.970000     24.700000       

 Data Cleaning & Target Variable
 

In [7]:
# ── Step 1: Create binary target ───────────────────────────────────────
# We ONLY keep loans with a definitive outcome
# "Current", "Late", "In Grace Period" are ambiguous — exclude them
keep_statuses = ['Fully Paid', 'Charged Off']
df = df[df['loan_status'].isin(keep_statuses)].copy()

# 1 = Default (Charged Off), 0 = Good (Fully Paid)
df['default'] = (df['loan_status'] == 'Charged Off').astype(int)

print(f"Default rate: {df['default'].mean():.2%}")
print(f"Remaining loans: {len(df):,}")
# Expect ~15-20% default rate — realistic for consumer credit

Default rate: 0.78%
Remaining loans: 2,450


In [8]:
# ── Step 2: Drop columns with too much missingness (>40%) ──────────────
missing_rate = df.isnull().mean()
cols_to_drop = missing_rate[missing_rate > 0.40].index.tolist()
print(f"\nDropping {len(cols_to_drop)} high-missing columns: {cols_to_drop[:10]}...")
df.drop(columns=cols_to_drop, inplace=True)




Dropping 44 high-missing columns: ['id', 'member_id', 'url', 'desc', 'mths_since_last_delinq', 'mths_since_last_record', 'next_pymnt_d', 'mths_since_last_major_derog', 'annual_inc_joint', 'dti_joint']...


In [9]:
# ── Step 3: Drop columns that leak future information ──────────────────
# These columns are only known AFTER the loan is issued/defaulted
# Including them would give the model "cheat" information — a critical
# mistake in credit modelling called "data leakage"
leakage_cols = [
    'total_pymnt', 'total_rec_prncp', 'total_rec_int',
    'total_rec_late_fee', 'recoveries', 'collection_recovery_fee',
    'last_pymnt_amnt', 'out_prncp', 'out_prncp_inv',
    'loan_status'   # ← already encoded as 'default'
]
leakage_cols = [c for c in leakage_cols if c in df.columns]
df.drop(columns=leakage_cols, inplace=True)



In [10]:
# ── Step 4: Handle remaining missing values ────────────────────────────
# For numeric columns: fill with median (robust to outliers)
# For categorical columns: fill with 'Unknown'
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(include=['object']).columns

df[num_cols] = df[num_cols].fillna(df[num_cols].median())
df[cat_cols] = df[cat_cols].fillna('Unknown')



/var/folders/dz/fm8cmd7j6wldjxj4w9tc1ss40000gn/T/ipykernel_13864/3851195635.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns


In [11]:
# ── Step 5: Convert % strings to floats ───────────────────────────────
# int_rate and revol_util come as strings like "13.5%"
for col in ['int_rate', 'revol_util']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.replace('%', '').astype(float)

print(f"\nFinal shape after cleaning: {df.shape}")
df.to_csv('loan_clean.csv', index=False)
print("Saved to loan_clean.csv")


Final shape after cleaning: (2450, 92)
Saved to loan_clean.csv


Feature Engineering (The Risk Indicators)

In [12]:
# ─────────────────────────────────────────────────────────────────────
# RISK INDICATOR 1: Debt-to-Income Ratio (DTI)
# Already in the data, but let's validate and cap outliers.
# DTI > 40% is considered high risk in consumer lending.
# ─────────────────────────────────────────────────────────────────────
# Cap at 99th percentile to remove outliers that would distort the model
dti_cap = df['dti'].quantile(0.99)
df['dti_capped'] = df['dti'].clip(upper=dti_cap)

# Risk flag: high DTI borrower (DTI > 30%)
df['high_dti_flag'] = (df['dti_capped'] > 30).astype(int)

print("RI 1 - DTI Distribution:")
print(df.groupby('high_dti_flag')['default'].mean().rename("default_rate"))




RI 1 - DTI Distribution:
high_dti_flag
0    0.008178
1    0.004016
Name: default_rate, dtype: float64


In [13]:
# ─────────────────────────────────────────────────────────────────────
# RISK INDICATOR 2: Loan-to-Income Ratio (LTI)
# How large is the loan relative to the borrower's annual income?
# A $50K loan on $40K income is very different from $50K on $200K income.
# ─────────────────────────────────────────────────────────────────────
df['loan_to_income'] = df['loan_amnt'] / (df['annual_inc'] + 1)  # +1 avoids div/0

# Cap at 99th percentile
lti_cap = df['loan_to_income'].quantile(0.99)
df['loan_to_income'] = df['loan_to_income'].clip(upper=lti_cap)

print("\nRI 2 - Loan-to-Income (top decile default rate):")
df['lti_decile'] = pd.qcut(df['loan_to_income'], q=10, labels=False, duplicates='drop')
print(df.groupby('lti_decile')['default'].mean())





RI 2 - Loan-to-Income (top decile default rate):
lti_decile
0    0.004065
1    0.004082
2    0.008197
3    0.012245
4    0.008163
5    0.004049
6    0.008230
7    0.004082
8    0.016327
9    0.008163
Name: default, dtype: float64


In [14]:
# ─────────────────────────────────────────────────────────────────────
# RISK INDICATOR 3: Employment Length (Stability Proxy)
# Longer employment = more income stability = lower default risk.
# emp_length comes as strings like "10+ years", "< 1 year", "n/a"
# ─────────────────────────────────────────────────────────────────────
emp_map = {
    '< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3,
    '4 years': 4, '5 years': 5, '6 years': 6, '7 years': 7,
    '8 years': 8, '9 years': 9, '10+ years': 10, 'n/a': -1, 'Unknown': -1
}
df['emp_length_num'] = df['emp_length'].map(emp_map).fillna(-1)

# Flag: unstable employment (< 2 years or unknown)
df['unstable_employment'] = (df['emp_length_num'] < 2).astype(int)

print("\nRI 3 - Employment Stability Default Rates:")
print(df.groupby('unstable_employment')['default'].mean())





RI 3 - Employment Stability Default Rates:
unstable_employment
0    0.004929
1    0.016026
Name: default, dtype: float64


In [15]:
# ─────────────────────────────────────────────────────────────────────
# RISK INDICATOR 4: Revolving Utilization Ratio
# Already in data as revol_util. This measures how much of available
# revolving credit (credit cards) the borrower is using.
# High utilization (>80%) signals financial stress.
# ─────────────────────────────────────────────────────────────────────
revol_cap = df['revol_util'].quantile(0.99)
df['revol_util_capped'] = df['revol_util'].clip(upper=revol_cap)

# Risk bucket: low (<30%), medium (30-80%), high (>80%)
df['revol_util_bucket'] = pd.cut(
    df['revol_util_capped'],
    bins=[-1, 30, 80, 200],
    labels=['Low', 'Medium', 'High']
)
print("\nRI 4 - Revolving Utilization Default Rates:")
print(df.groupby('revol_util_bucket', observed=True)['default'].mean())



RI 4 - Revolving Utilization Default Rates:
revol_util_bucket
Low       0.007005
Medium    0.005410
High      0.025126
Name: default, dtype: float64


In [16]:
# ─────────────────────────────────────────────────────────────────────
# RISK INDICATOR 5: Interest Rate Tier
# Higher interest rate = lender already assessed this as riskier.
# int_rate is a strong proxy for internal risk grade.
# ─────────────────────────────────────────────────────────────────────
df['int_rate_tier'] = pd.cut(
    df['int_rate'],
    bins=[0, 10, 15, 20, 100],
    labels=['Prime (<10%)', 'Near-Prime (10-15%)', 'Subprime (15-20%)', 'Deep Subprime (>20%)']
)
print("\nRI 5 - Interest Rate Tier Default Rates:")
print(df.groupby('int_rate_tier', observed=True)['default'].mean())

# ── Save engineered dataset ────────────────────────────────────────────
df.to_csv('loan_engineered.csv', index=False)
print(f"\nSaved engineered dataset. Shape: {df.shape}")


RI 5 - Interest Rate Tier Default Rates:
int_rate_tier
Prime (<10%)            0.008897
Near-Prime (10-15%)     0.008658
Subprime (15-20%)       0.006515
Deep Subprime (>20%)    0.005714
Name: default, dtype: float64

Saved engineered dataset. Shape: (2450, 101)
